# SIAF — Gasto Devengado: documentación del dataset

**Proyecto:** El Sol Que Más Rinde
**Fuente de datos:** Presupuesto y Ejecución de Gasto — Devengado Mensual, MEF
`datosabiertos.mef.gob.pe/dataset/presupuesto-y-ejecucion-de-gasto-devengado-mensual`

Este documento explica qué archivos descargamos, qué variables usamos de cada uno, qué significa cada una, y cómo las vamos a combinar para construir la tabla final `gasto_anemia` a nivel **ubigeo + Año**. La idea es que cualquiera del equipo pueda leer esto y entender el dataset sin tener que abrir el CSV.

---

## 1. Qué archivos descargamos y por qué

El MEF publica un CSV por año. La estructura cambia a partir de 2025:

| Años | Archivo | Estructura |
|---|---|---|
| 2021, 2022, 2023, 2024 | `YYYY-Gasto-Devengado.csv` | Un archivo por año, publicado como "anual" |
| 2025 | `2025-Gasto-Devengado-Mensual.csv` | Un archivo, pero con desglose mensual real |

Descargamos del 2021 al 2025 porque es el rango en el que también tenemos datos de anemia (SIEN) para hacer el cruce.

Existe también `2025-Gasto-Devengado-Diario.csv` (y su equivalente 2026), pero **no lo usamos**: es la misma información con otra frecuencia de actualización, no otra granularidad temporal útil para nosotros.

**Fase contable usada: Devengado.** El MEF reporta varias fases (Certificación, Compromiso, Devengado, Girado). Usamos Devengado porque es la fase en la que el bien o servicio ya fue recibido y la obligación de pago ya está reconocida — es el estándar para medir "cuánto se gastó realmente", no solo lo planeado ni lo pagado.

---

## 2. El problema de fondo: 2021–2024 no traen desglose mensual real

El diccionario oficial del MEF (`Gasto_Devengado_Diccionario.csv`) describe columnas mensuales (`MONTO_DEVENGADO_ENERO` … `MONTO_DEVENGADO_DICIEMBRE`) como si existieran en todos los años. En la práctica:

- **2021 a 2024:** las columnas mensuales vienen en cero. El único monto confiable es `MONTO_DEVENGADO_ANUAL`.
- **2025:** sí trae el desglose mensual real, poblado correctamente.

Como nuestra unidad de análisis es **distrito-año** (no distrito-mes), esto en realidad no nos genera un problema grave — solo tenemos que normalizar los dos formatos hacia un único número anual por fila.

### Cómo normalizamos

```
si ANO_EJE está entre 2021 y 2024:
    monto_devengado_anual = MONTO_DEVENGADO_ANUAL

si ANO_EJE == 2025:
    monto_devengado_anual = suma(MONTO_DEVENGADO_ENERO ... MONTO_DEVENGADO_DICIEMBRE)
```

Al final, todos los años quedan expresados en la misma columna: `monto_devengado_anual`, sin importar de qué archivo original vino la fila.

**Control de calidad sugerido:** para 2025, comparar la suma de los 12 meses contra `MONTO_DEVENGADO_ANUAL` (si ese campo también viene poblado ese año) — deberían coincidir. Si no coinciden, es señal de que hay que revisar el archivo antes de confiar en él.

---

## 3. Cómo se construye el UBIGEO

El CSV no trae un campo único de "ubigeo". Hay que armarlo concatenando tres códigos:

```
ubigeo = DEPARTAMENTO_EJECUTORA + PROVINCIA_EJECUTORA + DISTRITO_EJECUTORA
```

Cada uno es un código de 2 dígitos, así que el ubigeo final queda con 6 dígitos, igual que en RENAMU y en las demás tablas del proyecto (`centro_salud`, `personal`, `anemia_muni`).

⚠️ **Punto importante que hay que tener presente en todo el proyecto:**

Este ubigeo identifica **dónde está ubicada la entidad que ejecuta el gasto** (la unidad ejecutora), no necesariamente el distrito donde se presta el servicio. Por ejemplo, una unidad ejecutora de salud regional puede estar registrada en la capital de provincia y administrar gasto que en realidad se ejecuta en varios distritos rurales alrededor.

Esto es un riesgo ya identificado en el documento maestro del proyecto. Revisamos si existía una alternativa a nivel de la "meta" (la actividad/obra específica dentro del programa presupuestal), pero el campo de ubicación de la meta (`DEPARTAMENTO_META`) solo llega a nivel departamento, no distrito — así que no resuelve el problema.

**Decisión del equipo:** por ahora usamos `DISTRITO_EJECUTORA` como proxy del distrito de intervención, y lo dejamos declarado explícitamente en la pantalla de limitaciones de la aplicación final, tal como ya estaba planteado en el documento maestro.

---

## 4. Variables que sí usamos, y qué significa cada una

Nos quedamos solo con las variables directamente relacionadas con identificar el gasto en anemia y ubicarlo en el tiempo y el espacio. Todo lo demás (clasificación contable detallada, identificadores de la entidad, clasificación funcional, etc.) no aporta a nuestro nivel de análisis y lo descartamos.

| Variable | Qué significa | Para qué la usamos |
|---|---|---|
| `ANO_EJE` | Año de ejecución del presupuesto | Es la mitad de nuestra llave de unión (`ubigeo + Año`), igual que en RENAMU |
| `DEPARTAMENTO_EJECUTORA` | Código de departamento donde está ubicada la entidad que ejecuta el gasto | Primeros 2 dígitos del ubigeo |
| `PROVINCIA_EJECUTORA` | Código de provincia donde está ubicada la entidad | Dígitos 3-4 del ubigeo |
| `DISTRITO_EJECUTORA` | Código de distrito donde está ubicada la entidad | Dígitos 5-6 del ubigeo |
| `DISTRITO_EJECUTORA_NOMBRE` | Nombre del distrito | No entra al modelo — sirve solo para verificar visualmente que el ubigeo se armó bien y para detectar errores al momento de revisar el cruce |
| `PROGRAMA_PPTO` | Código del Programa Presupuestal | Con este código filtramos exactamente qué gasto es "gasto en anemia": PAN (0001) y los demás programas que toquen anemia (salud materno-neonatal, saneamiento rural, Cuna Más, Qali Warma, JUNTOS, incentivos municipales) |
| `PROGRAMA_PPTO_NOMBRE` | Nombre del programa presupuestal | Verificación de que filtramos el código correcto — también útil para poder decir en el pitch "de qué programa viene la plata" |
| `MONTO_DEVENGADO_ANUAL` | Monto total ejecutado en fase Devengado durante el año | Nuestra variable de gasto para 2021–2024 |
| `MONTO_DEVENGADO_ENERO` … `MONTO_DEVENGADO_DICIEMBRE` | Desglose mensual del gasto Devengado | Solo confiable en 2025 — se suman para construir el `monto_devengado_anual` equivalente de ese año |

**Variables que descartamos explícitamente** (identificadores de la entidad como `SECTOR`, `PLIEGO`, `EJECUTORA`; clasificación funcional como `FUNCION`, `DIVISION_FUNCIONAL`; clasificación contable como `GENERICA`, `SUBGENERICA`, `ESPECIFICA`; otras fases como `MONTO_CERTIFICADO_ANUAL`, `MONTO_COMPROMETIDO_ANUAL`, `MONTO_GIRADO_ANUAL`): no se relacionan directamente con identificar cuánto se gastó en anemia, dónde y cuándo — que es todo lo que necesita el modelo causal.

---

## 5. Cómo queda la tabla final

Después del filtro por `PROGRAMA_PPTO` (programas de anemia) y la normalización anual, agregamos por distrito-año, porque una misma combinación ubigeo-año puede tener varias filas (distintas entidades ejecutoras, distintas fuentes de financiamiento, etc. dentro del mismo programa):

```
gasto_anemia = groupby(['ubigeo', 'ANO_EJE'])['monto_devengado_anual'].sum()
```

Resultado esperado: una fila por distrito-año, con el monto total devengado en programas relacionados a anemia, lista para unirse con `centro_salud`, `personal` y `anemia_muni` usando `ubigeo + Año`.

---

## 6. Pendiente por resolver antes de cerrar esta etapa

- [ ] Confirmar la lista final y los códigos exactos de `PROGRAMA_PPTO` a incluir (PAN 0001 + complementarios)
- [ ] Correr el control de calidad de 2025 (suma de meses vs. anual, si aplica)
- [ ] Revisar cuántos distritos quedan con gasto = 0 después del join, como primera señal del tamaño del problema de `DISTRITO_EJECUTORA` vs. distrito de intervención real

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path


def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"

RAW_DIR = DATA / "raw" / "SIAF_gasto_devengado"

COLS_BASE = ['ANO_EJE', 'DEPARTAMENTO_EJECUTORA', 'PROVINCIA_EJECUTORA',
             'DISTRITO_EJECUTORA', 'DISTRITO_EJECUTORA_NOMBRE',
             'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']

MESES = ['MONTO_DEVENGADO_ENERO', 'MONTO_DEVENGADO_FEBRERO', 'MONTO_DEVENGADO_MARZO',
         'MONTO_DEVENGADO_ABRIL', 'MONTO_DEVENGADO_MAYO', 'MONTO_DEVENGADO_JUNIO',
         'MONTO_DEVENGADO_JULIO', 'MONTO_DEVENGADO_AGOSTO', 'MONTO_DEVENGADO_SEPTIEMBRE',
         'MONTO_DEVENGADO_OCTUBRE', 'MONTO_DEVENGADO_NOVIEMBRE', 'MONTO_DEVENGADO_DICIEMBRE']

archivos = {
    2021: RAW_DIR / "2021-Gasto-Devengado" / "2021-Gasto-Devengado.csv",
    2022: RAW_DIR / "2022-Gasto-Devengado" / "2022-Gasto-Devengado.csv",
    2023: RAW_DIR / "2023-Gasto-Devengado" / "2023-Gasto-Devengado.csv",
    2024: RAW_DIR / "2024-Gasto-Devengado" / "2024-Gasto-Devengado.csv",
    2025: RAW_DIR / "2025-Gasto-Devengado-Mensual" / "2025-Gasto-Devengado-Mensual.csv",
}


In [2]:
dfs = []

for anio, ruta in archivos.items():
    cols = COLS_BASE + (['MONTO_DEVENGADO_ANUAL'] if anio <= 2024 else MESES)
    df = pd.read_csv(ruta, sep=",", dtype=str, usecols=cols)

    # ubigeo
    df['ubigeo'] = (df['DEPARTAMENTO_EJECUTORA'].str.zfill(2)
                     + df['PROVINCIA_EJECUTORA'].str.zfill(2)
                     + df['DISTRITO_EJECUTORA'].str.zfill(2))

    # monto anual: directo para 2021-2024, sumando meses para 2025
    if anio <= 2024:
        df['monto_devengado_anual'] = pd.to_numeric(df['MONTO_DEVENGADO_ANUAL'], errors='coerce')
    else:
        df['monto_devengado_anual'] = df[MESES].apply(pd.to_numeric, errors='coerce').sum(axis=1)

    dfs.append(df[['ANO_EJE', 'ubigeo', 'DISTRITO_EJECUTORA_NOMBRE',
                    'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE', 'monto_devengado_anual']])

gasto_siaf = pd.concat(dfs, ignore_index=True)
gasto_siaf['ANO_EJE'] = gasto_siaf['ANO_EJE'].astype(int)

print(gasto_siaf.shape)
gasto_siaf.head()

(13419402, 6)


,ANO_EJE,ubigeo,DISTRITO_EJECUTORA_NOMBRE,PROGRAMA_PPTO,PROGRAMA_PPTO_NOMBRE,monto_devengado_anual
0,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
1,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
2,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
3,2021,150101,LIMA,9001,ACCIONES CENTRALES,3129.80
4,2021,150101,LIMA,9001,ACCIONES CENTRALES,7923.24


In [4]:
geobase_path = DATA / "clean" / "staging" / "geobase_distrital.gpkg"
geobase = gpd.read_file(geobase_path)

# Diagnóstico rápido: confirmar el nombre real de la columna de ubigeo en la geobase
print(geobase.columns.tolist())
print(f"Distritos en geobase: {len(geobase)}")

['UBIGEO', 'departamento', 'provincia', 'distrito', 'region', 'macroregion_inei', 'macroregion_minsa', 'capital', 'latitude', 'longitude', 'altitude', 'superficie', 'pob_densidad_2020', 'geometry']
Distritos en geobase: 1889


In [5]:
COL_UBIGEO_GEOBASE = "UBIGEO"  # ajustar según lo que confirme la celda anterior

validacion = gasto_siaf[['ubigeo']].drop_duplicates().merge(
    geobase[[COL_UBIGEO_GEOBASE]],
    left_on='ubigeo', right_on=COL_UBIGEO_GEOBASE, how='left', indicator=True
)

print(validacion['_merge'].value_counts())

no_cruzan = validacion.loc[validacion['_merge'] == 'left_only', 'ubigeo']
print(f"\n{len(no_cruzan)} ubigeos del SIAF no encontrados en la geobase:")
print(no_cruzan.tolist())

_merge
both          1889
left_only        3
right_only       0
Name: count, dtype: int64

3 ubigeos del SIAF no encontrados en la geobase:
['180107', '130112', '160405']


In [6]:
gasto_siaf[gasto_siaf['ubigeo'] == '160405'][['ubigeo', 'DISTRITO_EJECUTORA_NOMBRE']].drop_duplicates()

# Santa Rosa de Loreto (creado 3 julio 2025, Ley N° 32403, separado de Yavarí).
# El shapefile de referencia de INEI es anterior a esa fecha, por eso no lo incluye.
# Ver decisión del equipo en la celda de exclusión más abajo.


,ubigeo,DISTRITO_EJECUTORA_NOMBRE
11641131,160405,SANTA ROSA DE LORETO


In [7]:
# Celda 6 — Cuánto gasto representa ese distrito
# ============================================================
gasto_siaf[gasto_siaf['ubigeo'] == '160405']['monto_devengado_anual'].sum()
# Monto pequeño frente al total nacional -> se excluye, no se reasigna.

np.float64(402358.7)

In [8]:
# Celda 7 — Programas presupuestales presentes en la data
# ============================================================
programas = gasto_siaf[['PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']].drop_duplicates().sort_values('PROGRAMA_PPTO')
print(programas)

output_programas = RAW_DIR / "programas_presupuestales_unicos.csv"
programas.to_csv(output_programas, index=False, sep=";")
print(f"Guardado: {len(programas)} programas -> {output_programas}")

# PENDIENTE (punto 6 del propio notebook, todavía sin cerrar):
# confirmar la lista final de PROGRAMA_PPTO a incluir en gasto_anemia además de PAN (0001)
# -- salud materno-neonatal, saneamiento rural, Cuna Más, Qali Warma, JUNTOS, incentivos municipales.
# Mientras no se confirme esa lista, gasto_anemia_pan se queda solo con 0001 (ver celda siguiente).


        PROGRAMA_PPTO                               PROGRAMA_PPTO_NOMBRE
8240             0001                    PROGRAMA ARTICULADO NUTRICIONAL
8808             0002                             SALUD MATERNO NEONATAL
9102             0016                                       TBC-VIH/SIDA
7777             0017                ENFERMEDADES METAXENICAS Y ZOONOSIS
7888             0018                      ENFERMEDADES NO TRANSMISIBLES
...               ...                                                ...
7517403          0151  REDUCCION DE LA CORRUPCION EN EL USO DE LOS RE...
787459           1001  PRODUCTOS ESPECIFICOS PARA DESARROLLO INFANTIL...
6321             1002  PRODUCTOS ESPECIFICOS PARA REDUCCION DE LA VIO...
0                9001                                 ACCIONES CENTRALES
221              9002  ASIGNACIONES PRESUPUESTARIAS QUE NO RESULTAN E...

[93 rows x 2 columns]
Guardado: 93 programas -> c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\raw\SIAF_gasto_dev

In [11]:
gasto_siaf = pd.concat(dfs, ignore_index=True)
gasto_siaf['ANO_EJE'] = gasto_siaf['ANO_EJE'].astype(int)

# Reduce memoria: estas columnas tienen pocos valores únicos repetidos millones de veces
for col in ['ubigeo', 'DISTRITO_EJECUTORA_NOMBRE', 'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']:
    gasto_siaf[col] = gasto_siaf[col].astype('category')

print(gasto_siaf.shape)
print(gasto_siaf.memory_usage(deep=True).sum() / 1e6, "MB")
gasto_siaf.head()

(13419402, 6)
295.291109 MB


,ANO_EJE,ubigeo,DISTRITO_EJECUTORA_NOMBRE,PROGRAMA_PPTO,PROGRAMA_PPTO_NOMBRE,monto_devengado_anual
0,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
1,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
2,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
3,2021,150101,LIMA,9001,ACCIONES CENTRALES,3129.80
4,2021,150101,LIMA,9001,ACCIONES CENTRALES,7923.24


In [12]:
n_antes = len(gasto_siaf)
monto_excluido = gasto_siaf.loc[gasto_siaf['ubigeo'] == '160405', 'monto_devengado_anual'].sum()

gasto_siaf = gasto_siaf[gasto_siaf['ubigeo'] != '160405']  # sin .copy()

print(f"Filas excluidas: {n_antes - len(gasto_siaf)}")
print(f"Monto excluido: S/ {monto_excluido:,.2f}")


Filas excluidas: 67
Monto excluido: S/ 402,358.70


In [13]:
# ============================================================
# Celda 9 — Agregar a nivel distrito-año: gasto total y gasto en anemia (PAN)
# ============================================================
gasto_total = (
    gasto_siaf
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_total'})
)

# Gasto en anemia: solo PAN (0001) por ahora -- ver pendiente en celda 7
gasto_anemia = (
    gasto_siaf[gasto_siaf['PROGRAMA_PPTO'] == '0001']
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_anemia_pan'})
)

gasto_final = gasto_total.merge(gasto_anemia, on=['ANO_EJE', 'ubigeo'], how='left')
gasto_final['gasto_anemia_pan'] = gasto_final['gasto_anemia_pan'].fillna(0)

print(f"Filas (año-ubigeo): {len(gasto_final):,}")
gasto_final.head()

Filas (año-ubigeo): 9,452


,ANO_EJE,ubigeo,gasto_total,gasto_anemia_pan
0,2021,010101,7.452997e+08,44747568.78
1,2021,010102,9.765828e+05,0.00
2,2021,010103,1.303335e+06,16060.00
3,2021,010104,2.132187e+06,0.00
4,2021,010105,1.380533e+06,5584.48


In [14]:
# ============================================================
# Celda 10 — Cobertura por año, y confirmar que 160405 ya no está
# ============================================================
resumen_anual = gasto_final.groupby('ANO_EJE')['ubigeo'].nunique()
print(resumen_anual)
print(f"\nTotal filas: {len(gasto_final):,}")
print(f"Total distritos únicos: {gasto_final['ubigeo'].nunique()}")
print(f"\n¿Sigue 160405 en gasto_final?: {'160405' in gasto_final['ubigeo'].values}")


ANO_EJE
2021    1889
2022    1890
2023    1891
2024    1891
2025    1891
Name: ubigeo, dtype: int64

Total filas: 9,452
Total distritos únicos: 1891

¿Sigue 160405 en gasto_final?: False


In [15]:
# ============================================================
# Celda 11 — Panel balanceado: qué combinaciones año-ubigeo faltan
# ============================================================
todos_los_ubigeos = gasto_final['ubigeo'].unique()
años = gasto_final['ANO_EJE'].unique()

panel_completo = pd.MultiIndex.from_product([años, todos_los_ubigeos], names=['ANO_EJE', 'ubigeo']).to_frame(index=False)

faltantes = panel_completo.merge(gasto_final[['ANO_EJE', 'ubigeo']], on=['ANO_EJE', 'ubigeo'], how='left', indicator=True)
faltantes = faltantes[faltantes['_merge'] == 'left_only']

print(faltantes)


      ANO_EJE  ubigeo     _merge
1889     2021  180107  left_only
1890     2021  130112  left_only
3781     2022  130112  left_only


In [16]:
# ============================================================
# Celda 12 (NUEVA) — Diagnóstico de los faltantes
# Revisa si esos ubigeos existen en OTROS años, y busca su nombre de distrito
# para decidir si es ausencia real de ejecución o un problema de la fuente.
# ============================================================
ubigeos_faltantes = faltantes['ubigeo'].unique()

for ubigeo in ubigeos_faltantes:
    años_con_data = gasto_final.loc[gasto_final['ubigeo'] == ubigeo, 'ANO_EJE'].tolist()
    nombre = gasto_siaf.loc[gasto_siaf['ubigeo'] == ubigeo, 'DISTRITO_EJECUTORA_NOMBRE'].drop_duplicates().tolist()
    print(f"ubigeo {ubigeo} -> nombre(s): {nombre} | aparece en años: {años_con_data}")

# PENDIENTE: con esto se puede ver si el distrito simplemente no tuvo ninguna fila
# en el CSV crudo del SIAF para ese año (ausencia real de reporte/ejecución) o si es
# un problema puntual de la fuente. Falta decidir si se dejan en NaN, se imputan en 0,
# o se documentan como dato faltante en la pantalla de limitaciones.


ubigeo 180107 -> nombre(s): ['SAN ANTONIO'] | aparece en años: [2022, 2023, 2024, 2025]
ubigeo 130112 -> nombre(s): ['ALTO TRUJILLO'] | aparece en años: [2023, 2024, 2025]


In [17]:
# ============================================================
# Celda 13 (NUEVA) — Guardado final en data/clean/staging/
# ============================================================
anio_min = gasto_final['ANO_EJE'].min()
anio_max = gasto_final['ANO_EJE'].max()

output_path = DATA / "clean" / "staging" / f"siaf_gasto_devengado_distrital_{anio_min}_{anio_max}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

gasto_final.to_csv(output_path, index=False)

print(f"Guardado: {output_path}")
print(gasto_final.shape)

Guardado: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\staging\siaf_gasto_devengado_distrital_2021_2025.csv
(9452, 4)


In [42]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw/SIAF_gasto_devengado")

COLS_BASE = ['ANO_EJE', 'DEPARTAMENTO_EJECUTORA', 'PROVINCIA_EJECUTORA',
             'DISTRITO_EJECUTORA', 'DISTRITO_EJECUTORA_NOMBRE',
             'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']

MESES = ['MONTO_DEVENGADO_ENERO','MONTO_DEVENGADO_FEBRERO','MONTO_DEVENGADO_MARZO',
         'MONTO_DEVENGADO_ABRIL','MONTO_DEVENGADO_MAYO','MONTO_DEVENGADO_JUNIO',
         'MONTO_DEVENGADO_JULIO','MONTO_DEVENGADO_AGOSTO','MONTO_DEVENGADO_SEPTIEMBRE',
         'MONTO_DEVENGADO_OCTUBRE','MONTO_DEVENGADO_NOVIEMBRE','MONTO_DEVENGADO_DICIEMBRE']

archivos = {
    2021: RAW_DIR / "2021-Gasto-Devengado" / "2021-Gasto-Devengado.csv",
    2022: RAW_DIR / "2022-Gasto-Devengado" / "2022-Gasto-Devengado.csv",
    2023: RAW_DIR / "2023-Gasto-Devengado" / "2023-Gasto-Devengado.csv",
    2024: RAW_DIR / "2024-Gasto-Devengado" / "2024-Gasto-Devengado.csv",
    2025: RAW_DIR / "2025-Gasto-Devengado-Mensual" / "2025-Gasto-Devengado-Mensual.csv",
}

In [43]:
dfs = []

for anio, ruta in archivos.items():
    cols = COLS_BASE + (['MONTO_DEVENGADO_ANUAL'] if anio <= 2024 else MESES)
    df = pd.read_csv(ruta, sep=",", dtype=str, usecols=cols)

    # ubigeo
    df['ubigeo'] = (df['DEPARTAMENTO_EJECUTORA'].str.zfill(2)
                     + df['PROVINCIA_EJECUTORA'].str.zfill(2)
                     + df['DISTRITO_EJECUTORA'].str.zfill(2))

    # monto anual: directo para 2021-2024, sumando meses para 2025
    if anio <= 2024:
        df['monto_devengado_anual'] = pd.to_numeric(df['MONTO_DEVENGADO_ANUAL'], errors='coerce')
    else:
        df['monto_devengado_anual'] = df[MESES].apply(pd.to_numeric, errors='coerce').sum(axis=1)

    dfs.append(df[['ANO_EJE', 'ubigeo', 'DISTRITO_EJECUTORA_NOMBRE',
                    'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE', 'monto_devengado_anual']])

gasto_siaf = pd.concat(dfs, ignore_index=True)
gasto_siaf['ANO_EJE'] = gasto_siaf['ANO_EJE'].astype(int)

gasto_siaf.head()

,ANO_EJE,ubigeo,DISTRITO_EJECUTORA_NOMBRE,PROGRAMA_PPTO,PROGRAMA_PPTO_NOMBRE,monto_devengado_anual
0,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
1,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
2,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
3,2021,150101,LIMA,9001,ACCIONES CENTRALES,3129.80
4,2021,150101,LIMA,9001,ACCIONES CENTRALES,7923.24


Queremos validar q realmente el ubigeo anterior lo es porque lo habíamos construido

In [47]:
validacion = gasto_siaf[['ubigeo']].drop_duplicates().merge(
    inei[['UBIGEO']],  # <- reemplaza 'UBIGEO' por el nombre real que viste arriba
    left_on='ubigeo', right_on='UBIGEO', how='left', indicator=True
)

print(validacion['_merge'].value_counts())

no_cruzan = validacion.loc[validacion['_merge'] == 'left_only', 'ubigeo']
print(f"\n{len(no_cruzan)} ubigeos del SIAF no encontrados en INEI:")
print(no_cruzan.tolist())

_merge
both          1891
left_only        1
right_only       0
Name: count, dtype: int64

1 ubigeos del SIAF no encontrados en INEI:
['160405']


In [48]:
gasto_siaf[gasto_siaf['ubigeo'] == '160405'][['ubigeo', 'DISTRITO_EJECUTORA_NOMBRE']].drop_duplicates()

,ubigeo,DISTRITO_EJECUTORA_NOMBRE
11641131,160405,SANTA ROSA DE LORETO


Santa Rosa (a veces registrado como "Santa Rosa de Loreto") es un distrito de la provincia de Maynas, departamento de Loreto, creado en años recientes. Es exactamente el escenario que ya habían anotado en el documento maestro como riesgo conocido ("control de distritos creados después de 2017").

Validación del ubigeo contra INEI

Se construyó el ubigeo del SIAF (`DEPARTAMENTO_EJECUTORA + PROVINCIA_EJECUTORA + DISTRITO_EJECUTORA`)
y se cruzó contra el shapefile oficial de límites distritales de INEI (2025 CPV).

**Resultado: 1891 de 1892 ubigeos únicos (99.9%) cruzan correctamente.**

El único que no cruza es `160405` — **Santa Rosa de Loreto**, provincia de Mariscal Ramón Castilla,
departamento de Loreto. Este distrito fue creado el 3 de julio de 2025 (Ley N° 32403), separándose
del distrito de Yavarí. El shapefile de INEI usado como referencia fue generado antes de esa fecha,
por lo que no lo incluye.

**Decisión:** no es un error del pipeline ni del ubigeo construido — es un distrito legítimamente
nuevo que el shapefile de referencia todavía no contempla. Se documenta como limitación conocida.
Para efectos del modelo, el gasto de `160405` puede: (a) excluirse de la muestra por no tener
polígono ni contexto satelital propio, o (b) reasignarse temporalmente a Yavarí (su distrito de
origen) si el volumen de gasto es significativo. Pendiente decidir con el equipo.

In [ ]:
gasto_siaf[gasto_siaf['ubigeo'] == '160405']['monto_devengado_anual'].sum()
# veamos cuanto era el monto devengado en ese distrito q no toamremos en cuenta, es una cifra pequeña asi q no le haremos caso

np.float64(402358.7)

**Decisión del equipo:** se excluye `160405` (Santa Rosa de Loreto) de `gasto_siaf`.
Monto excluido: S/ 402,358.70 (2021-2025, todos los programas presupuestales) — marginal
frente al total nacional. Además, al no existir en el shapefile de INEI usado, tampoco
sería posible construir su contexto satelital para el modelo causal, así que conservar
su gasto no aportaría una observación utilizable de todas formas.

In [54]:
# cuales son los porgramas?
programas = gasto_siaf[['PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']].drop_duplicates().sort_values('PROGRAMA_PPTO')
print(programas)

        PROGRAMA_PPTO                               PROGRAMA_PPTO_NOMBRE
8240             0001                    PROGRAMA ARTICULADO NUTRICIONAL
8808             0002                             SALUD MATERNO NEONATAL
9102             0016                                       TBC-VIH/SIDA
7777             0017                ENFERMEDADES METAXENICAS Y ZOONOSIS
7888             0018                      ENFERMEDADES NO TRANSMISIBLES
...               ...                                                ...
7517403          0151  REDUCCION DE LA CORRUPCION EN EL USO DE LOS RE...
787459           1001  PRODUCTOS ESPECIFICOS PARA DESARROLLO INFANTIL...
6321             1002  PRODUCTOS ESPECIFICOS PARA REDUCCION DE LA VIO...
0                9001                                 ACCIONES CENTRALES
221              9002  ASIGNACIONES PRESUPUESTARIAS QUE NO RESULTAN E...

[93 rows x 2 columns]


In [55]:
programas.to_csv("../data/raw/SIAF_gasto_devengado/programas_presupuestales_unicos.csv",
                  index=False, sep=";")

print(f"Guardado: {len(programas)} programas")

Guardado: 93 programas


In [56]:
# Gasto total: todos los programas presupuestales, sin filtrar
gasto_total = (
    gasto_siaf
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_total'})
)

# Gasto en anemia: solo PAN (0001)
gasto_anemia = (
    gasto_siaf[gasto_siaf['PROGRAMA_PPTO'] == '0001']
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_anemia_pan'})
)

# Unimos ambos en una sola tabla, una fila por año-ubigeo
gasto_final = gasto_total.merge(gasto_anemia, on=['ANO_EJE', 'ubigeo'], how='left')

# Si un distrito no tuvo gasto en PAN ese año, el merge deja NaN — lo correcto es 0, no vacío
gasto_final['gasto_anemia_pan'] = gasto_final['gasto_anemia_pan'].fillna(0)

print(f"Filas (año-ubigeo): {len(gasto_final):,}")
gasto_final.head()

Filas (año-ubigeo): 9,453


,ANO_EJE,ubigeo,gasto_total,gasto_anemia_pan
0,2021,010101,7.452997e+08,44747568.78
1,2021,010102,9.765828e+05,0.00
2,2021,010103,1.303335e+06,16060.00
3,2021,010104,2.132187e+06,0.00
4,2021,010105,1.380533e+06,5584.48


In [57]:
# Cuántos distritos únicos hay por año
resumen_anual = gasto_final.groupby('ANO_EJE')['ubigeo'].nunique()
print(resumen_anual)

print(f"\nTotal filas: {len(gasto_final):,}")
print(f"Total distritos únicos (todos los años): {gasto_final['ubigeo'].nunique()}")

ANO_EJE
2021    1889
2022    1890
2023    1891
2024    1891
2025    1892
Name: ubigeo, dtype: int64

Total filas: 9,453
Total distritos únicos (todos los años): 1892


In [58]:
'160405' in gasto_final['ubigeo'].values

True

In [59]:
# 1. Excluir Santa Rosa de Loreto (creado julio 2025, no existe en el shapefile INEI de referencia)
n_antes = len(gasto_siaf)
monto_excluido = gasto_siaf.loc[gasto_siaf['ubigeo'] == '160405', 'monto_devengado_anual'].sum()

gasto_siaf = gasto_siaf[gasto_siaf['ubigeo'] != '160405'].copy()

print(f"Filas excluidas: {n_antes - len(gasto_siaf)}")
print(f"Monto excluido: S/ {monto_excluido:,.2f}")

# 2. Volver a generar gasto_total, gasto_anemia y el merge final
gasto_total = (
    gasto_siaf
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_total'})
)

gasto_anemia = (
    gasto_siaf[gasto_siaf['PROGRAMA_PPTO'] == '0001']
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_anemia_pan'})
)

gasto_final = gasto_total.merge(gasto_anemia, on=['ANO_EJE', 'ubigeo'], how='left')
gasto_final['gasto_anemia_pan'] = gasto_final['gasto_anemia_pan'].fillna(0)

# 3. Verificar que ya no está, y que el total de distritos bajó a 1891
print('160405' in gasto_final['ubigeo'].values)
print(f"Distritos únicos: {gasto_final['ubigeo'].nunique()}")
print(gasto_final.groupby('ANO_EJE')['ubigeo'].nunique())

Filas excluidas: 67
Monto excluido: S/ 402,358.70
False
Distritos únicos: 1891
ANO_EJE
2021    1889
2022    1890
2023    1891
2024    1891
2025    1891
Name: ubigeo, dtype: int64


In [60]:
# ¿Cuáles son los distritos que no aparecen en todos los años?
todos_los_ubigeos = gasto_final['ubigeo'].unique()
años = gasto_final['ANO_EJE'].unique()

panel_completo = pd.MultiIndex.from_product([años, todos_los_ubigeos], names=['ANO_EJE', 'ubigeo']).to_frame(index=False)

faltantes = panel_completo.merge(gasto_final[['ANO_EJE', 'ubigeo']], on=['ANO_EJE', 'ubigeo'], how='left', indicator=True)
faltantes = faltantes[faltantes['_merge'] == 'left_only']

print(faltantes)

      ANO_EJE  ubigeo     _merge
1889     2021  180107  left_only
1890     2021  130112  left_only
3781     2022  130112  left_only
